# 07 · Metals

A metal is the same `embed`, given an arrangement. `rx.metal(source, geometry)` enumerates the distinct
coordination isomers; pick one and embed it. Examples use ionic dative SMILES, with donor arrows pointing
toward the metal and anionic donors carrying formal charges. `rx.metal` also accepts neutral/covalent
Lewis forms such as `N->[Pt](Cl)Cl`; the preferred written form is `N->[Pt+2](<-[Cl-])<-[Cl-]`.

The metal is swapped to a bond-less surrogate for the force field, because UFF cannot type a bonded
transition metal, and restored with its oxidation state and dative bonds on the way out.


In [ ]:
import rxembed as rx
import xyzrender
from rdkit import RDLogger
from rdkit.Chem import rdMolTransforms as T

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

## Coordination isomers: the permutations

Square-planar MA₂B₂ gives cis and trans; octahedral MA₃B₃ gives mer and fac. `.summary()` prints each
per-vertex arrangement and Λ/Δ label. Pick one with `.select(index=...)`, `.select(arrangement=...)`, or
`.select(chirality=...)`. Arrangement is unambiguous when several isomers share a coarse label.

In [ ]:
print("square-planar Pd (cis/trans):")
rx.metal("CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC", "square_planar").summary()
print("\noctahedral Co(NH3)3Cl3 (mer/fac):")
rx.metal("N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]", "octahedral").summary()

The fac Co isomer, its coordination sphere drawn as dotted dative bonds.


In [ ]:
co = rx.metal("N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]", "octahedral").select(
    index=1
)  # the fac isomer (index 1 in .summary())
ens = rx.embed(co).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    report.assert_ok()
xyzrender.render(ens.mol, no_hy=True, idx="s")

The trans Pd isomer: the two Cl sit about 180° apart across the metal.


In [ ]:
trans = rx.metal("CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC", "square_planar").select(index=1)  # the trans isomer (index 1)
ens = rx.embed(trans).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    report.assert_ok()
cl = [d for d in trans.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "Cl"]
print(f"{len(ens)} conformers | Cl-Pd-Cl = {ens.measure((cl[0], trans.metal, cl[1]))['mean']:.0f}° (trans)")
xyzrender.render(
    ens.mol, no_hy=True, labels=[f"{cl[0] + 1} {trans.metal + 1} {cl[1] + 1} a"]
)  # label the trans Cl-Pd-Cl angle

## Across coordination geometries

The same API covers linear, tetrahedral, square-planar, octahedral, trigonal-bipyramidal and
square-pyramidal geometries.

In [ ]:
for g, smi in {
    "linear": "[Cl-]->[Au+]<-[Cl-]",
    "tetrahedral": "[Cl-]->[Zn+2](<-[Cl-])(<-[Cl-])<-[Cl-]",
    "square_planar": "CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC",
    "octahedral": "N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]",
}.items():
    isos = rx.metal(smi, g)
    print(f"  {g:14s}: {len(isos)} isomer(s) {[i.label for i in isos]}")

## Metal-centre chirality

Λ and Δ, from a point-group parity construction rather than RDKit's native metal stereo, whose permutation
is not order-invariant for equivalent ligands.


In [ ]:
isos = rx.metal("[Cl-]->[Co+2]12(<-[Cl-])(<-NCCN->1)<-NCCN->2", "octahedral")  # [Co(en)2Cl2]; bis-chelate + 2 Cl
isos.summary()  # trans | achiral ;  cis | Δ ;  cis | Λ
delta = isos.select(chirality="Δ")  # pick the Δ enantiomer by its handedness (name-agnostic, unambiguous)
ens = rx.embed(delta, n=8).minimize().mc(preset="ensemble").minimize().prune()  # relax -> search -> re-relax -> dedup
reps = ens.representatives()
print(f"cis-Δ [Co(en)2Cl2]: {ens.n} conformers -> {reps.n} distinct mode(s)")
for report in reps.check(donors=list(delta.donors)).values():
    report.assert_ok()  # includes folded-donor checks
xyzrender.render(reps.mol, no_hy=True, idx="s")

## Chelating ligands: bidentate

A chelate's donors sit on one ligand fragment; rxembed detects same-fragment donors and widens their L–M–L window (a rigid backbone bite is distorted off the ideal) while still separating cis from trans. Ethylenediamine on Pd (a 5-membered chelate ring, N–Pd–N bite ~82°) + two chloride:

In [ ]:
en = rx.metal("[Cl-]->[Pd+2]1(<-[Br-])<-NCCN->1", "square_planar")  # the NCCN ring pins both N to Pd
iso = en.select()
ens = rx.embed(iso, n=10).minimize().mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    report.assert_ok()
nn = [d for d in iso.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "N"]
print(f"{len(ens)} conformers | en bite N-Pd-N = {ens.measure((nn[0], iso.metal, nn[1]))['mean']:.0f}°")
xyzrender.render(ens.mol, no_hy=True)

## Side-on η² coordination

A directly π-bonded donor pair binds side-on as one haptic site. rxembed holds the donor bond while applying
both metal distances. The Ni example below keeps its coordinated C=C intact.

In [ ]:
iso = rx.metal(
    "COC(=O)[C]12->[Ni+2]3(<-[O-]C(=O)C(c4ccccc4)[N-]->3c3ccccc3)<-[C]=1(C(=O)OC)C2(C)C(C)(C)C",
    "square_planar",
).select(index=0)
pi = next(  # the directly π-bonded donor pair: the side-on C=C
    (a, b)
    for a in iso.donors
    for b in iso.donors
    if a < b and (bd := iso.mol.GetBondBetweenAtoms(a, b)) and bd.GetBondTypeAsDouble() >= 2
)
ens = rx.embed(iso, n=8).minimize().mc(preset="ensemble").minimize().prune()
ens.filter("geometry")
c = ens.mol.GetConformer(ens.ids[0])
print(
    f"side-on C=C {pi}: {ens.n} confs | Ni-C {T.GetBondLength(c, iso.metal, pi[0]):.2f}/"
    f"{T.GetBondLength(c, iso.metal, pi[1]):.2f} Å (both bind) | C=C {T.GetBondLength(c, *pi):.2f} Å (held intact)"
)
xyzrender.render(ens.mol, no_hy=True, labels=[f"{iso.metal + 1} {pi[0] + 1} d", f"{iso.metal + 1} {pi[1] + 1} d"])

## Vacant sites: a coordination pocket

Name a geometry with more vertices than donors and the empty vertex is left as a pocket (`·` in the arrangement) for a substrate to bind later. A 3-donor Pd in a square-planar field leaves one site open.

In [ ]:
pocket = rx.metal("CC#[N]->[Pd+2](<-[Cl-])<-[Cl-]", "square_planar")  # N + 2 Cl = 3 donors, 4 vertices
pocket.summary()  # '·' marks the vacant coordination site
iso = pocket.select(index=0)  # the cis pocket (both Cl cis; the '·' vacancy sits opposite the lone Cl)
ens = rx.embed(iso).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    report.assert_ok()
xyzrender.render(ens.mol, no_hy=True, idx="s")  # the '·' vacancy sits opposite the lone Cl